# USEPA 2018-2019 Join + Trim
The purpose of this notebook is to combine the datasets provided by the National Rivers & Streams Assessment (NRSA) from 2018-2019 that include land, physiographic division designation, chemical, physical habitat, and information about >1,000 sites across the U.S. Only the 2018-1019 dataset was used for model building given the wealth of watershed characteristics (elevation, land use, soil,etc) assigned to each site. The data was manually renamed for joining and clarity, but the README in the data folder will provide more information on data specifics.

E. Reilly Oare

In [1]:
# Read in libs
import os  
import warnings
import pandas as pd 
import numpy as np 

## Step 1. Read in Data
We will need to read in five datasets:
- information (metadata essentially)
- physical habitat
- landscape
- chemical
- physiographic province

In [2]:
# Define base directory
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))

In [4]:
# Define paths to datasets
info_path = os.path.join(BASE_DIR, 'data', 'raw', 'national', 'nrsa_1819_siteinfo_renamed.xlsx')
phab_path = os.path.join(BASE_DIR, 'data', 'raw', 'national', 'nrsa_1819_phab_renamed.xlsx')
land_path = os.path.join(BASE_DIR, 'data', 'raw', 'national', 'nrsa_1819_land_renamed.xlsx')
chem_path = os.path.join(BASE_DIR, 'data', 'raw', 'national', 'nrsa_1819_chem_renamed.xlsx')
physio_path = os.path.join(BASE_DIR, 'data', 'raw', 'national', 'epa_1819_physio.xlsx')

# Read in 2018-2019 datasets
epa_1819_siteinfo = pd.read_excel(info_path)
epa_1819_phab = pd.read_excel(phab_path)
epa_1819_land = pd.read_excel(land_path)
epa_1819_chem = pd.read_excel(chem_path)
epa_physio = pd.read_excel(physio_path).drop_duplicates(subset=['Site ID'], keep='last')
epa_physio = epa_physio[['Site ID', 'LAT_DD83', 'LON_DD83', 'DIVISION']]

## Step 2. Drop rows without the target
Since we are trying to predict embeddedness, we only need datapoints that include that value.

In [5]:
# Select for only datapoints with target value, Mean streambed embeddedness (%)
epa_1819_emd = epa_1819_phab.dropna(subset = ['Mean streambed embeddedness (%)'], axis = 0)

In [6]:
# Print number of rows
print("Shape of epa 1819:", epa_1819_emd.shape)

Shape of epa 1819: (1225, 80)


## Step 3. Merge 2018-2019 datasets
Pandas .merge method is used to combine all five datasets.

In [7]:
# Find first set of common columns
common1819_1 = list(set(epa_1819_emd.columns).intersection(epa_1819_siteinfo.columns))

# Merge 2018 - 2019 embeddedness with site info data
epa_1819_mrg1 = epa_1819_emd.merge(epa_1819_siteinfo,
                                  on = common1819_1,
                                  how = 'left')
# Display first join stats
epa_1819_mrg1.describe()

,Unique ID,Collection Date,Year,Mean Bank Angle (degrees),Median Bank Angle (Degrees),Mean Undercut Distance (m),Mid-channel Canopy Density (%),Bank Canopy Density (%),% Glide Habitat,Pools - % of Reach,...,Mean streambed embeddedness (%),Embeddedness Center of Channel (Mean % wetted streambed),Mean Residual Depth at thalweg (cm).1,Ground layer Vegetated areal cover,Riparian Veg Quality Index--Scaled XCMGW,Riparian Veg Quality Index--Scaled XCDENBK,Riparian Disturbance index (scaled W1_Hall 0-1),Riparian Disturbance index (1 - RDIST1),Riparian Quality Index (Scaled 0 to 1),Elevation
count,1.225000e+03,1225,1225.000000,1225.000000,1225.000000,1139.000000,1225.000000,1225.000000,1224.000000,1224.000000,...,1225.000000,1225.000000,1213.000000,1225.000000,1225.000000,1225.000000,1225.000000,1225.000000,1225.000000,1225.000000
mean,2.013531e+06,2019-01-02 19:12:00,2018.447347,32.553811,28.198367,0.497280,49.117398,74.917489,54.600779,18.430052,...,66.948291,61.612097,17.050245,0.636490,0.451977,0.774257,0.395091,0.604909,0.574920,687.308286
min,2.012488e+06,2018-05-07 00:00:00,2018.000000,2.272727,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.400000,0.000000,0.213121,0.006818,0.100000,0.100000,0.000000,0.145266,0.129308,1.380000
25%,2.012985e+06,2018-07-23 00:00:00,2018.000000,22.954545,17.500000,0.000000,14.304813,61.904762,22.712121,0.000000,...,47.090909,38.030303,7.484060,0.468182,0.284602,0.657143,0.185185,0.400000,0.439457,197.050000
50%,2.013512e+06,2018-09-17 00:00:00,2018.000000,31.045455,25.000000,0.011364,52.272727,83.422460,56.000000,4.000000,...,69.454545,62.878788,13.660134,0.664773,0.440057,0.850802,0.440685,0.559315,0.581222,363.600000
75%,2.014057e+06,2019-07-14 00:00:00,2019.000000,40.863636,36.500000,0.047727,82.486631,94.607843,89.083333,26.683168,...,90.181818,89.242424,22.745961,0.835227,0.603182,0.951471,0.600000,0.814815,0.721877,1005.660000
max,2.014737e+06,2019-10-02 00:00:00,2019.000000,113.181818,132.500000,141.500000,102.807487,101.069519,100.000000,100.000000,...,100.000000,100.000000,141.038119,1.227273,1.000000,1.009626,0.854734,1.000000,0.995973,3276.825000
std,6.328987e+02,NaN,0.497423,13.866381,15.124367,5.791739,34.794648,24.781463,34.961941,27.214523,...,25.630079,29.129316,13.675148,0.238789,0.217883,0.223033,0.243086,0.243086,0.186350,724.895574


In [8]:
# Merge 2018 - 2019 embeddedness with land data
epa_1819_mrg2 = epa_1819_mrg1.merge(epa_1819_land,
                                  on = 'Site ID',
                                  how = 'left')

# Display second join stats
epa_1819_mrg2.describe()

,Unique ID_x,Collection Date,Year,Mean Bank Angle (degrees),Median Bank Angle (Degrees),Mean Undercut Distance (m),Mid-channel Canopy Density (%),Bank Canopy Density (%),% Glide Habitat,Pools - % of Reach,...,Horton overland flow,Watershed mean of the high values of permeability,Mean of all permeability values,Site mean of the high values of depth to bedrock,Watershed mean of the high values of depth to bedrock,Estimated watershed annual runoff,Mean of all sand values within watershed,Mean of all S values within watershed,Mean Composite Topographic Index (CTI)[Wetness Index],Watershed area in square kilometers
count,1.225000e+03,1225,1225.000000,1225.000000,1225.000000,1139.000000,1225.000000,1225.000000,1224.000000,1224.000000,...,1222.000000,1225.000000,1225.000000,1.225000e+03,1225.000000,1225.000000,1225.000000,1225.000000,1225.000000,1225.000000
mean,2.013531e+06,2019-01-02 19:12:00,2018.447347,32.553811,28.198367,0.497280,49.117398,74.917489,54.600779,18.430052,...,6.540776,4.444544,7.405846,-2.775510e+35,52.341973,333.624844,34.351431,0.334457,791.383590,1533.704598
min,2.012488e+06,2018-05-07 00:00:00,2018.000000,2.272727,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.201697,0.315354,-3.400000e+38,19.035720,0.054716,2.798677,0.006253,544.146192,0.212400
25%,2.012985e+06,2018-07-23 00:00:00,2018.000000,22.954545,17.500000,0.000000,14.304813,61.904762,22.712121,0.000000,...,1.194227,2.080263,3.428585,4.960000e+01,46.790070,66.877075,23.070000,0.043150,701.607200,12.510000
50%,2.013512e+06,2018-09-17 00:00:00,2018.000000,31.045455,25.000000,0.011364,52.272727,83.422460,56.000000,4.000000,...,4.000000,3.173474,5.220109,5.853368e+01,53.794081,288.124103,32.493488,0.121141,781.144817,61.115400
75%,2.014057e+06,2019-07-14 00:00:00,2019.000000,40.863636,36.500000,0.047727,82.486631,94.607843,89.083333,26.683168,...,8.008025,5.513705,9.163084,6.000000e+01,59.891278,511.917725,43.962259,0.297838,863.675559,354.512700
max,2.014737e+06,2019-10-02 00:00:00,2019.000000,113.181818,132.500000,141.500000,102.807487,101.069519,100.000000,100.000000,...,64.839696,19.650399,45.230809,7.251000e+01,72.354710,3731.739057,92.040000,23.662374,1413.590351,143417.842200
std,6.328987e+02,NaN,0.497423,13.866381,15.124367,5.791739,34.794648,24.781463,34.961941,27.214523,...,8.178867,3.616567,6.218882,9.714286e+36,8.029661,324.280581,16.837861,1.179754,127.913982,8190.236711


In [9]:
# Find third set of common columns
common1819_3 = list(set(epa_1819_mrg2.columns).intersection(epa_1819_chem.columns))

# Merge 2018 - 2019 embeddedness with chem data
epa_1819_mrg3 = epa_1819_mrg2.merge(epa_1819_chem,
                                  on = common1819_3,
                                  how = 'left')

# Print third 2018 - 2019 dataframe
epa_1819_mrg3.describe()

,Unique ID_x,Collection Date,Year,Mean Bank Angle (degrees),Median Bank Angle (Degrees),Mean Undercut Distance (m),Mid-channel Canopy Density (%),Bank Canopy Density (%),% Glide Habitat,Pools - % of Reach,...,Dissolved Total Nitrogen (mg N/L),Total Nitrogen (mg N/L),pH,Potassium (mg/L),Dissolved Total Phosphorus (ug/L),Total Phosphorus (ug/L),Silica (mg/L),Sodium (mg/L),Sulfate (mg/L),Turbidity (NTU)
count,1.225000e+03,1225,1225.000000,1225.000000,1225.000000,1139.000000,1225.000000,1225.000000,1224.000000,1224.000000,...,1124.000000,1223.000000,1223.000000,1224.000000,1127.000000,1223.000000,1224.000000,1224.000000,1223.000000,1224.000000
mean,2.013531e+06,2019-01-02 19:12:00,2018.447347,32.553811,28.198367,0.497280,49.117398,74.917489,54.600779,18.430052,...,1.111102,1.329302,7.920164,4.580556,81.498411,139.380392,13.355583,137.539788,212.333097,31.745496
min,2.012488e+06,2018-05-07 00:00:00,2018.000000,2.272727,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.013000,0.022000,3.700000,0.039000,1.259000,-2.000000,0.093000,0.195000,0.037000,0.010000
25%,2.012985e+06,2018-07-23 00:00:00,2018.000000,22.954545,17.500000,0.000000,14.304813,61.904762,22.712121,0.000000,...,0.210000,0.263000,7.640000,1.016500,10.010000,21.425000,6.793000,3.435250,3.240500,1.730000
50%,2.013512e+06,2018-09-17 00:00:00,2018.000000,31.045455,25.000000,0.011364,52.272727,83.422460,56.000000,4.000000,...,0.480000,0.620000,8.090000,2.252000,21.368000,50.966000,10.894500,9.727000,11.951000,4.915000
75%,2.014057e+06,2019-07-14 00:00:00,2019.000000,40.863636,36.500000,0.047727,82.486631,94.607843,89.083333,26.683168,...,1.037500,1.343000,8.300000,4.751250,53.253000,117.768000,16.472500,36.403250,47.912500,14.017500
max,2.014737e+06,2019-10-02 00:00:00,2019.000000,113.181818,132.500000,141.500000,102.807487,101.069519,100.000000,100.000000,...,21.150000,21.175000,9.640000,169.309000,4612.175000,4667.413000,63.414000,29200.000000,33548.530000,4230.000000
std,6.328987e+02,NaN,0.497423,13.866381,15.124367,5.791739,34.794648,24.781463,34.961941,27.214523,...,2.078531,2.229225,0.569202,9.311450,286.741894,360.751027,9.927683,1240.227688,1412.837059,196.555573


In [10]:
# Find third set of common columns
common1819_4 = list(set(epa_1819_mrg3.columns).intersection(epa_physio.columns))

# Merge 2018 - 2019 embeddedness with physiographic division dataset
epa_1819_mrg4 = epa_1819_mrg3.merge(epa_physio,
                                  on = common1819_4,
                                  how = 'left')

# Print fourth 2018 - 2019 dataframe
epa_1819_mrg4

,Unique ID_x,Site ID,Collection Date,Year,Mean Bank Angle (degrees),Median Bank Angle (Degrees),Mean Undercut Distance (m),Mid-channel Canopy Density (%),Bank Canopy Density (%),Constraining Features,...,Dissolved Total Phosphorus (ug/L),Total Phosphorus (ug/L),Silica (mg/L),Sodium (mg/L),Sulfate (mg/L),Total Suspended Solids (mg/L),Turbidity (NTU),LAT_DD83,LON_DD83,DIVISION
0,2014737,NRS18_OK_10062,2018-06-20,2018,27.727273,20.0,0.000000,100.000000,100.000000,TERRACE,...,NaN,223.000,30.500,55.900,20.400,264,123.00,35.545389,-99.722705,INTERIOR PLAINS
1,2012488,NRS18_OK_10431,2018-06-25,2018,33.590909,23.5,0.000000,74.064171,76.737968,TERRACE,...,NaN,55.000,6.650,23.700,365.000,23,15.80,36.902168,-95.164255,INTERIOR PLAINS
2,2012489,NRS18_OK_10025,2018-07-02,2018,19.318182,10.0,0.000000,12.032086,67.112299,HILLSLOPE,...,NaN,50.000,8.650,6.550,8.200,18,15.60,33.929449,-96.070242,ATLANTIC PLAIN
3,2012490,NRS18_WI_10091,2018-07-02,2018,31.954545,30.0,0.042857,52.941176,79.144385,HUMAN,...,NaN,199.000,15.800,9.640,22.200,8,7.68,42.536868,-89.395560,INTERIOR PLAINS
4,2012491,NRS18_WI_10006,2018-07-12,2018,27.823529,20.0,2.076471,52.058824,97.213622,TERRACE,...,NaN,17.900,10.800,3.060,7.230,NaN,2.29,45.496257,-87.982493,LAURENTIAN UPLAND
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1220,2014708,NRS18_PA_10009,2019-09-25,2019,22.227273,21.0,0.000000,10.294118,78.342246,HUMAN,...,5.783,10.433,3.300,8.452,30.529,0.8,1.40,41.922584,-77.129228,APPALACHIAN HIGHLANDS
1221,2014714,NRS18_SD_10005,2019-09-25,2019,26.166667,20.0,0.169583,38.848039,59.803922,TERRACE,...,18.619,85.764,17.504,115.192,1727.158,95.3,68.43,43.070132,-98.800398,INTERIOR PLAINS
1222,2014715,NRS18_NC_10897,2019-09-24,2019,57.523810,60.0,0.000000,51.871658,87.165775,TERRACE,...,282.605,340.111,18.387,50.055,23.809,3.9,3.14,35.996645,-80.336835,APPALACHIAN HIGHLANDS
1223,2014720,NRS18_ND_10049,2019-09-27,2019,35.523810,35.0,0.065000,0.000000,70.868347,NOCONST,...,136.981,217.090,14.030,207.517,492.481,17.3,20.77,46.799719,-101.106841,INTERIOR PLAINS


## Step 4. Clean/Trim EPA Dataset
We will check for columns that have >5% of their data missing. We will analyze if these are able to be removed based on professional opinion.

In [14]:
# Find % nulls in each columns
missing_1819 = epa_1819_mrg4.isnull().mean()

# Display columns that are missing > 5% of their data
missing_1819[missing_1819 > 0.05].index

Index(['Mean Undercut Distance (m)',
       'Mean Slope of water surface-Map-derived measurement (%)',
       'Nitrite (mg N/L)', 'Dissolved Total Nitrogen (mg N/L)',
       ' Dissolved Total Phosphorus (ug/L)'],
      dtype='object')

In [15]:
# Drop columns where the missing percentages are more than 5% of the column (0.05)
epa_1819_cln = epa_1819_mrg4.drop(columns = missing_1819[missing_1819 > 0.05].index)

In [16]:
# Print new datasets
epa_1819_cln.columns

Index(['Unique ID_x', 'Site ID', 'Collection Date', 'Year',
       'Mean Bank Angle (degrees)', 'Median Bank Angle (Degrees)',
       'Mid-channel Canopy Density (%)', 'Bank Canopy Density (%)',
       'Constraining Features', '% Glide Habitat',
       ...
       'Potassium (mg/L)', 'Total Phosphorus (ug/L)', 'Silica (mg/L)',
       'Sodium (mg/L)', 'Sulfate (mg/L)', 'Total Suspended Solids (mg/L)',
       'Turbidity (NTU)', 'LAT_DD83', 'LON_DD83', 'DIVISION'],
      dtype='object', length=167)

## Step 5. Add physics-informed attribute
We're going to add bankfull shear velocity by using the bankfull width and bankfull width to depth ratio. The equation for $u^*$ is:
$$u^* = \sqrt{g * R_h * S}$$
Where $g$ is gravity,
<br> $R_h$ is hydraulic radius (assuming rectangular channel), and
<br> $S$ is slope.

In [17]:
# Define variables
g = 9.81 # m/s^2
W_bkfl = epa_1819_cln['Mean Bankfull Width (m)']
H_bkfl = epa_1819_cln['Mean Bankfull Width (m)'] / epa_1819_cln['Bankfull Width/Depth Ratio']
S = epa_1819_cln['Mean Slope of water surface (%)'] / 100

# Do calculations
A = W_bkfl * H_bkfl
P = W_bkfl + (2 * H_bkfl)
R_h = A / P
epa_1819_cln['Bankfull Shear Velocity (m/s)'] = np.sqrt(g * R_h * S)

# Display number of null valus
epa_1819_cln[['Bankfull Shear Velocity (m/s)']].isnull().sum()

Bankfull Shear Velocity (m/s)    7
dtype: int64

## Step 6. Export Trimmed Dataset
The combined dataset will go to the next stage of EDA.

In [19]:
# Push trimmed dataset to processed data folder
trimmed_path = os.path.join(BASE_DIR, "data", "processed", "national", "epa_1819_trimmed.xlsx")
epa_1819_cln.to_excel(trimmed_path)
print(f"Dataset saved to {trimmed_path}")

Dataset saved to C:\Users\reill\Documents\Virginia Tech\Czuba Research\Embeddedness\GitHub\Embeddedness_ML_XAI\data\processed\national\epa_1819_trimmed.xlsx
